In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import joblib
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (accuracy_score,precision_score,recall_score,f1_score,confusion_matrix,classification_report)
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
BASE_DIR=Path.cwd().parent
DATA_PATH=BASE_DIR / "data" / "processed" / "priority_prediction_dataset.csv"
df=pd.read_csv(DATA_PATH)

In [3]:
df.head()

,Severity,Disaster_Type,People_Count,Children_Count,Elderly_Count,Injured_Count,Road_Accessibility,Weather_Condition,Location_Type,Time_Since_Incident,Incident_Time,Infrastructure_Damage,Communication_Available,Evacuation_Required,Priority_Score,Priority,Incident_ID
0,0.83,Flood,32,10,3,11,Accessible,Rain,Coastal,74,Morning,Medium,Yes,Yes,40.88,Medium,INC-000001
1,0.89,Flood,84,22,24,21,Not Accessible,Thunderstorm,Urban,85,Morning,High,No,No,71.88,High,INC-000002
2,0.80,Earthquake,177,17,25,91,Not Accessible,Clear,Mountain,20,Evening,Medium,No,Yes,79.48,Critical,INC-000003
3,0.62,Flood,41,18,16,11,Not Accessible,Heavy Rain,Urban,37,Morning,Medium,No,Yes,49.68,Medium,INC-000004
4,0.65,Fire,41,0,9,11,Not Accessible,Rain,Forest,32,Afternoon,Low,Yes,Yes,39.37,Medium,INC-000005


In [5]:
print(df.shape)

(50000, 17)


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 17 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Severity                 50000 non-null  float64
 1   Disaster_Type            50000 non-null  object 
 2   People_Count             50000 non-null  int64  
 3   Children_Count           50000 non-null  int64  
 4   Elderly_Count            50000 non-null  int64  
 5   Injured_Count            50000 non-null  int64  
 6   Road_Accessibility       50000 non-null  object 
 7   Weather_Condition        50000 non-null  object 
 8   Location_Type            50000 non-null  object 
 9   Time_Since_Incident      50000 non-null  int64  
 10  Incident_Time            50000 non-null  object 
 11  Infrastructure_Damage    50000 non-null  object 
 12  Communication_Available  50000 non-null  object 
 13  Evacuation_Required      50000 non-null  object 
 14  Priority_Score        

In [7]:
df.describe(include="all")

,Severity,Disaster_Type,People_Count,Children_Count,Elderly_Count,Injured_Count,Road_Accessibility,Weather_Condition,Location_Type,Time_Since_Incident,Incident_Time,Infrastructure_Damage,Communication_Available,Evacuation_Required,Priority_Score,Priority,Incident_ID
count,50000.000000,50000,50000.000000,50000.000000,50000.000000,50000.000000,50000,50000,50000,50000.000000,50000,50000,50000,50000,50000.00000,50000,50000
unique,NaN,5,NaN,NaN,NaN,NaN,3,5,6,NaN,4,3,2,2,NaN,4,50000
top,NaN,Flood,NaN,NaN,NaN,NaN,Not Accessible,Heavy Rain,Urban,NaN,Night,High,No,Yes,NaN,High,INC-000001
freq,NaN,17586,NaN,NaN,NaN,NaN,29901,12650,15244,NaN,12591,23995,28742,40207,NaN,22717,1
mean,0.780938,NaN,82.621640,10.943940,10.720740,22.937300,NaN,NaN,NaN,63.381020,NaN,NaN,NaN,NaN,57.13132,NaN,NaN
std,0.136210,NaN,56.995972,8.566055,8.674769,20.118625,NaN,NaN,NaN,44.472804,NaN,NaN,NaN,NaN,18.34891,NaN,NaN
min,0.400000,NaN,5.000000,0.000000,0.000000,1.000000,NaN,NaN,NaN,1.000000,NaN,NaN,NaN,NaN,16.63000,NaN,NaN
25%,0.690000,NaN,37.000000,4.000000,4.000000,9.000000,NaN,NaN,NaN,26.000000,NaN,NaN,NaN,NaN,43.50000,NaN,NaN
50%,0.800000,NaN,68.000000,9.000000,8.000000,17.000000,NaN,NaN,NaN,55.000000,NaN,NaN,NaN,NaN,56.33000,NaN,NaN
75%,0.890000,NaN,123.000000,17.000000,17.000000,30.000000,NaN,NaN,NaN,96.000000,NaN,NaN,NaN,NaN,69.73000,NaN,NaN


In [8]:
X=df.drop(columns=["Incident_ID","Priority","Priority_Score"])
y=df["Priority"]

In [10]:
print("Feature Shape :",X.shape)
print("Target Shape:",y.shape)

Feature Shape : (50000, 14)
Target Shape: (50000,)


In [11]:
label_encoder=LabelEncoder()
y=label_encoder.fit_transform(y)
print(label_encoder.classes_)

['Critical' 'High' 'Low' 'Medium']


In [13]:
X=pd.get_dummies(X,drop_first=False)
feature_columns=X.columns.tolist()
print("Encoded Feature Shape:",X.shape)

Encoded Feature Shape: (50000, 36)


In [14]:
X.head()

,Severity,People_Count,Children_Count,Elderly_Count,Injured_Count,Time_Since_Incident,Disaster_Type_Cyclone,Disaster_Type_Earthquake,Disaster_Type_Fire,Disaster_Type_Flood,...,Incident_Time_Evening,Incident_Time_Morning,Incident_Time_Night,Infrastructure_Damage_High,Infrastructure_Damage_Low,Infrastructure_Damage_Medium,Communication_Available_No,Communication_Available_Yes,Evacuation_Required_No,Evacuation_Required_Yes
0,0.83,32,10,3,11,74,False,False,False,True,...,False,True,False,False,False,True,False,True,False,True
1,0.89,84,22,24,21,85,False,False,False,True,...,False,True,False,True,False,False,True,False,True,False
2,0.80,177,17,25,91,20,False,True,False,False,...,True,False,False,False,False,True,True,False,False,True
3,0.62,41,18,16,11,37,False,False,False,True,...,False,True,False,False,False,True,True,False,False,True
4,0.65,41,0,9,11,32,False,False,True,False,...,False,False,False,False,True,False,False,True,False,True


In [15]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.20,random_state=42,stratify=y)
print(X_train.shape)
print(X_test.shape)
print("training labels:",len(y_train))
print("testing labels :",len(y_test))

(40000, 36)
(10000, 36)
training labels: 40000
testing labels : 10000


In [22]:
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()
numeric_columns=["Severity","People_Count","Children_Count","Elderly_Count","Injured_Count","Time_Since_Incident"]
X_train[numeric_columns]=scaler.fit_transform(X_train[numeric_columns])
X_test[numeric_columns]=scaler.transform(X_test[numeric_columns])

In [26]:
lr=LogisticRegression(max_iter=5000,class_weight="balanced",random_state=42)
lr.fit(X_train,y_train)
lr_pred=lr.predict(X_test)

In [29]:
print(accuracy_score(y_test,lr_pred))
print(classification_report(y_test,lr_pred))

0.9515
              precision    recall  f1-score   support

           0       0.88      0.97      0.92      1811
           1       0.97      0.93      0.95      4543
           2       0.93      1.00      0.96       703
           3       0.97      0.97      0.97      2943

    accuracy                           0.95     10000
   macro avg       0.94      0.97      0.95     10000
weighted avg       0.95      0.95      0.95     10000



In [30]:
rf=RandomForestClassifier(n_estimators=300,random_state=42,class_weight="balanced")
rf.fit(X_train,y_train)
rf_pred=rf.predict(X_test)
print(accuracy_score(y_test,rf_pred))
print(classification_report(y_test,rf_pred))
      

0.9373
              precision    recall  f1-score   support

           0       0.94      0.91      0.93      1811
           1       0.92      0.96      0.94      4543
           2       0.96      0.93      0.94       703
           3       0.96      0.91      0.94      2943

    accuracy                           0.94     10000
   macro avg       0.95      0.93      0.94     10000
weighted avg       0.94      0.94      0.94     10000



In [32]:
xgb=XGBClassifier(n_estimators=300,learning_rate=0.1,max_depth=6,random_state=42,eval_metric="mlogloss")
xgb.fit(X_train,y_train)
xgb_pred=xgb.predict(X_test)
print(accuracy_score(y_test,xgb_pred))
print(classification_report(y_test,xgb_pred))

0.9684
              precision    recall  f1-score   support

           0       0.97      0.96      0.97      1811
           1       0.97      0.98      0.97      4543
           2       0.97      0.97      0.97       703
           3       0.97      0.96      0.97      2943

    accuracy                           0.97     10000
   macro avg       0.97      0.97      0.97     10000
weighted avg       0.97      0.97      0.97     10000



In [38]:
from pathlib import Path
import joblib
import json
MODEL_DIR=Path.cwd().parent / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(xgb,MODEL_DIR / "priority_model.pkl")

['C:\\Users\\HP\\n1\\ResqAI\\models\\priority_model.pkl']

In [39]:
joblib.dump(scaler,MODEL_DIR / "priority_scaler.pkl")

['C:\\Users\\HP\\n1\\ResqAI\\models\\priority_scaler.pkl']

In [42]:
joblib.dump(label_encoder,MODEL_DIR / "priority_label_encoder.pkl")


['C:\\Users\\HP\\n1\\ResqAI\\models\\priority_label_encoder.pkl']

In [43]:
with open(MODEL_DIR / "priority_feature_column.json","w") as f:
    json.dump(feature_columns, f, indent=4)

In [44]:
training_report={
    "best_model":"XGBoost","accuracy":float(accuracy_score(y_test, xgb_pred)),
    "weighted_f1":float(f1_score(y_test, xgb_pred, average="weighted")),
    "classes":list(label_encoder.classes_),
    "num_features":len(feature_columns),
    "train_samples":len(X_train),
    "test_samples":len(X_test)
}
with open(MODEL_DIR / "priority_training_report.json","w") as f:
    json.dump(training_report, f, indent=4)
print("done")

done


In [45]:
MODEL_DIR=Path.cwd().parent /"models"/"priority_prediction"
MODEL_DIR.mkdir(parents=True, exist_ok=True)